Cell 1 - Import

In [0]:
from pyspark.sql import functions as F
from datetime import datetime 
import json

CATALOG = "cat_flight_liability"
SILVER_SCHEMA = "db_silver"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.delay_flights_current"

GOLD_SCHEMA = "db_gold"
GOLD_LIABILITY_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.delay_liability"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

Cell 2 - Logging

In [0]:
def log(level , message , **kwargs):
    print(json.dumps({"timestamp" : datetime.utcnow().isoformat() , "level" : level , "message" : message , **kwargs}))

log("Info", "Gold pipeline run started")

Cell 3 - Read Silver

In [0]:
sivler_df = spark.table(SILVER_TABLE)
silver_count = silver_df.count()
log("INFO", "Reading silver table for gold aggregation", row_count = silver_count)

Cell 3 -   Financial Liability Aggregations

In [0]:


if silver_count == 0 :
    log("WARNING", "Silver table is empty.Skipping Gold Liability Calculations")
else:
    AVG_PASSENGERS_PER_FLIGHT = 0
    gold_financial_liability_df = (
        silver_df
        .withColumn(
            "compensation_tier",
            F.when(F.col("max_delay_minutes") < 60 , "no_compensation")
            .when(F.col("max_delay_minutes") < 180 , "tier_1_short")
            .when(F.col("max_delay_minutes") < 240 ,"tier_2_medium" )
            .otherwise("tier_3_long")
            )
        .withColumn(
            "payout_per_pax_inr",
            F.when(F.col(max_delay_minutes) < 60 ,F.lit(0))
            .when(F.col("max_delay_minutes") < 180 , F.lit(2500) )
            .when(F.col("max_delay_minutes") < 240 , F.lit(5000) )
            .otherwise(F.lit(7500))
        )
    )
    tier_counts = gold_financial_liability_df.groupBy("compensation_tier").count().collect()
    log("INFO","Compensation Tiers Assigned", tiers={row["compensation_tier"]: row["count"] for row in tier_counts})
    
    gold_financial_liability_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_LIABILITY_TABLE)
    log("INFO", "Gold table written", target=GOLD_LIABILITY_TABLE, row_count=gold_financial_liability_df.count())